In [3]:
!git clone https://github.com/lorenzo-stacchio/Deep-Armocromia.git

Cloning into 'Deep-Armocromia'...


In [ ]:
import os
import io
import time
import base64
import re
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from openai import OpenAI
from tqdm.notebook import tqdm
from PIL import Image

# ====================================================
# 1. 설정 (RunPod 엔드포인트 및 경로)
# ====================================================
# 런팟 대시보드에서 복사한 Proxy 주소를 입력하세요 (뒤에 /v1 필수)
RUNPOD_BASE_URL = "https://uuhv2p4jkmtr4r-8000.proxy.runpod.io/v1" 
API_KEY = "qwenbench123"
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

client = OpenAI(
    base_url=RUNPOD_BASE_URL,
    api_key=API_KEY
)

# 이미지 image_ade681.png의 구조를 반영한 경로 설정
BASE_DIR = Path("./Deep-Armocromia/dataset") # 데이터셋 핵심 폴더
ARMOCROMIA_CSV = BASE_DIR / "annotations.csv"
ARMOCROMIA_IMG_DIR = BASE_DIR / "images"

# ====================================================
# 2. Helper Functions
# ====================================================
def encode_image(img_input):
    """이미지 경로 또는 객체를 Base64로 변환"""
    if isinstance(img_input, (str, Path)):
        img = Image.open(img_input)
    else:
        img = img_input
        
    if img.mode != "RGB":
        img = img.convert("RGB")
        
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    return base64.b64encode(buf.getvalue()).decode()

def ask_vlm(question, img_input):
    img_b64 = encode_image(img_input)
    start = time.time()
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": question},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
                    ]
                }
            ],
            max_tokens=50,
            temperature=0,
            timeout=120
        )
        return response.choices[0].message.content.strip(), time.time() - start
    except Exception as e:
        print(f"\n[Error]: {e}")
        return "", 0.0

# ====================================================
# 3. 데이터셋 평가 함수들
# ====================================================
def eval_colorbench():
    print("🚀 [1/2] ColorBench 평가 시작...")
    ds = load_dataset("umd-zhou-lab/ColorBench", "color_extraction", split="test")
    correct, total = 0, len(ds)
    
    for row in tqdm(ds, desc="ColorBench"):
        answer = str(row["answer"]).lower()
        pred, _ = ask_vlm(row["question"], row["image"])
        if re.search(rf'\b{re.escape(answer)}\b', pred.lower()):
            correct += 1
    return (correct / total) * 100

def eval_armocromia():
    print("\n🚀 [2/2] Deep-Armocromia 평가 시작...")
    if not ARMOCROMIA_CSV.exists():
        print(f"⚠️ CSV를 찾을 수 없습니다: {ARMOCROMIA_CSV}")
        print("Deep-Armocromia 폴더 안에 'dataset' 폴더와 'annotations.csv'가 있는지 확인하세요.")
        return 0
        
    df = pd.read_csv(ARMOCROMIA_CSV)
    SEASON_MAP = {"primavera": "spring", "estate": "summer", "autunno": "autumn", "inverno": "winter"}
    
    PROMPT = (
        "You are an expert in Armocromia. Analyze the face and classify the seasonal palette "
        "into: Spring, Summer, Autumn, or Winter. Output ONLY the one word season name."
    )

    correct, total_valid = 0, 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Armocromia"):
        img_name = str(row["filename"]) if "filename" in df.columns else str(row.iloc[0])
        if not img_name.lower().endswith(('.jpg', '.png', '.jpeg')): img_name += ".jpg"
        
        img_path = ARMOCROMIA_IMG_DIR / img_name
        if not img_path.exists(): continue

        gt = SEASON_MAP.get(str(row["class"]).strip().lower(), "")
        pred, _ = ask_vlm(PROMPT, img_path)
        
        if re.search(rf'\b{re.escape(gt)}\b', pred.lower()):
            correct += 1
        total_valid += 1

    return (correct / total_valid) * 100 if total_valid > 0 else 0

# ====================================================
# 4. 실행 및 결과 출력
# ====================================================
if __name__ == "__main__":
    cb_acc = eval_colorbench()
    armo_acc = eval_armocromia()
    
    print("\n" + "="*40)
    print(f"📊 최종 결과 요약")
    print(f"- ColorBench 정답률: {cb_acc:.2f}%")
    print(f"- Armocromia 정답률: {armo_acc:.2f}%")
    print("="*40)